## 1. My lane as an ML task

My chosen lane is **Lane 2: Refresh / Content Opportunity Scoring**.

I frame this lane as a **ranking / scoring problem**. The goal is not only to predict whether a page is declining, but to rank pages by which ones should be reviewed first for possible content refresh.

The model or scoring system should produce a priority score for each page. Pages with higher scores should be reviewed earlier by a content or SEO team.

This is a ranking/scoring task because the real business question is not simply “Is this page declining?” The more useful question is: “Which pages should we inspect first when we have limited time?”

## 2. Target or proxy

The target/proxy I would use is:

`is_declining_label`

This label is created from the `trend_direction` column:

- `1` if `trend_direction == "down"`
- `0` otherwise

This is a proxy target because it does not directly prove that a page needs a refresh. A declining page may need a refresh, but it may also be declining for other reasons such as seasonality, ranking changes, search intent changes, or lower demand.

Even with that limitation, `is_declining_label` is useful because it gives a measurable signal that can help prioritize pages for review.

## 3. Success metric

My main success metric would be **Precision@50**.

Precision@50 measures how many of the top 50 recommended pages are actually declining pages.

This metric fits the task because a content team usually has limited review time. They may only be able to inspect the top 20, top 50, or top 100 pages. The model is useful if the highest-ranked pages contain many true review candidates.

I would prefer Precision@50 over general accuracy because the business goal is not to classify every page equally. The goal is to make the top of the review queue useful.

In [6]:
# Section 4: The unit of analysis as a real dataframe
# Lane: Refresh / Content Opportunity Scoring
# Unit of analysis: one row = one anonymized content page

import os
import subprocess
import pandas as pd
import numpy as np

# Clone starter repo in Colab if it is not already available
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

# Load starter dataset
data_path = os.path.join(REPO_DIR, "data", "raw", "content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

# Create target/proxy column
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Select columns relevant to Lane 2
lane_df = df[
    [
        "content_id",
        "content_type",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "word_count",
        "trend_direction",
        "is_declining_label",
    ]
].copy()

# Show the unit of analysis
print("Unit of analysis: one row = one anonymized content page")
print("Number of rows/pages:", len(lane_df))
print("Number of columns shown:", lane_df.shape[1])

lane_df.head(10)

Unit of analysis: one row = one anonymized content page
Number of rows/pages: 30000
Number of columns shown: 10


,content_id,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,keyword article,187,20,3803,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,keyword article,445,25,15320,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,keyword article,141,20,12581,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,keyword article,463,22,11751,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,keyword article,263,14,19140,44.0,0.13,2803.0,down,1
5,content_d4084a4bc775,keyword article,147,20,3970,8.5,0.03,3080.0,down,1
6,content_9a34b442b552,keyword article,90,20,20,7.0,0.00,3059.0,down,1
7,content_a63219c6e95a,keyword article,445,22,1724,21.2,0.06,NaN,stable,0
8,content_5e6c160719bc,keyword article,90,20,32574,46.0,0.09,3807.0,down,1
9,content_c27558df2b0c,keyword article,257,104,1240,4.9,0.16,NaN,down,1


The dataframe above shows the unit of analysis for my lane.

One row represents one anonymized content page. Each page has observable signals such as content type, content age, days since last update, impressions, average position, CTR, word count, and trend direction.

For my Lane 2 task, the target/proxy column is `is_declining_label`. A value of `1` means the page is marked as declining, while a value of `0` means it is not marked as declining.

The model or scoring system would use these page-level signals to rank which pages should be reviewed first for possible content refresh.

In [7]:
# Show target distribution

target_counts = lane_df["is_declining_label"].value_counts().rename(index={0: "not_declining", 1: "declining"})
target_rate = lane_df["is_declining_label"].mean()

print("Target distribution:")
print(target_counts)
print()
print("Declining rate:", round(target_rate, 3))

Target distribution:
is_declining_label
declining        16262
not_declining    13738
Name: count, dtype: int64

Declining rate: 0.542


The target distribution shows that 16,262 pages are labeled as declining and 13,738 pages are not declining. The declining rate is 0.542, meaning about 54.2% of the pages are marked as declining.

This target is useful for Lane 2 because the goal is to prioritize pages for content review or refresh. However, the label should be treated as a proxy. A declining page does not automatically prove that a refresh is required, but it gives a measurable signal that can help rank pages for human review.

## 5. Why ML beats a fixed rule here

A fixed rule might say:

“Review pages where days since last update is greater than 180 and impressions are greater than 500.”

That rule is simple and understandable, but it is limited because it only checks a few conditions. It treats every page the same way and may miss more complex patterns.

For Lane 2, refresh priority depends on multiple signals together. A page may be old but still performing well. Another page may be newer but have low CTR, declining trend, weak position, or strong impressions that make it worth reviewing.

ML or scoring is useful here because the goal is not just to say yes or no. The goal is to rank pages into a review queue so a content or SEO team can decide which pages to inspect first.

The output supports a real content action: review, refresh, improve, protect, prune, or monitor selected pages.

## 6. Self-check

- [x] I named my lane: Lane 2, Refresh / Content Opportunity Scoring.
- [x] I named the ML task type: ranking / scoring.
- [x] I named the target/proxy: `is_declining_label`.
- [x] I explained how the target is created from `trend_direction`.
- [x] I named a success metric: Precision@50.
- [x] I showed the unit of analysis as a real dataframe.
- [x] I explained that one row represents one anonymized content page.
- [x] I showed what the target column looks like using real counts.
- [x] I explained why ML/scoring is better than only using a fixed rule.
- [x] I tied the output to a real content action: review, refresh, improve, protect, prune, or monitor.